# AutoFuse使能基础

上一节介绍了 AutoFuse 的基本概念、接入路线和融合过程等，本节面向使用者说明如何开启 AutoFuse 的基础能力。这里的"使能"可以理解为：在执行模型图编译前，通过环境变量打开 AutoFuse 功能入口，并按需声明基础融合能力。学完本节后，你应能知道基础开关怎么打开、如何验证配置是否生效，以及哪些融合能力需要额外声明。更深入的融合原理、实践与问题定位会在后续章节展开。

本节学习大纲如下：

- 使用前提
- AutoFuse 使能配置
- 常见误区


## 1. 使用前提

使用 AutoFuse 前提条件如下：

<table align="left">
<thead><tr><th>条件</th><th>要求</th><th>说明</th></tr></thead>
<tbody>
<tr><td>安装软件包</td><td>准备带有 AI 处理器的硬件环境，并安装驱动、固件和 CANN 软件包</td><td>具体安装步骤请参见 <a href='https://www.hiascend.com/document/detail/zh/CANNCommunityEdition/910beta1/softwareinst/instg/instg_0000.html?OS=openEuler&InstallType=netyum'>《CANN 软件安装》</a></td></tr>
<tr><td>GCC 版本</td><td>9.5.0 及以上</td><td>建议使用 9.5.0 版本</td></tr>
<tr><td>CMake 版本</td><td>3.20.0 及以上</td><td>建议使用 3.20.0 版本</td></tr>
<tr><td>CANN 环境变量</td><td>安装 CANN 软件后，使用 CANN 运行用户登录环境，并执行 `source ${INSTALL_DIR}/set_env.sh`</td><td>`${INSTALL_DIR}` 请替换为 CANN 软件安装后的文件存储路径；以 root 用户安装为例，默认路径为 /usr/local/Ascend/cann</td></tr>
</tbody>
</table>
<div style="clear:left"></div>

完成上述准备后，再在同一个编译环境中配置 AutoFuse 基础开关。


## 2. AutoFuse 使能配置

AutoFuse 的使能方式与对接路线密切相关。使用 GE 图编译路线时，需要在模型图编译前通过环境变量 <code>AUTOFUSE_FLAGS</code> 开启自动融合；使用 PyTorch Inductor 对接路线时，当前无需额外配置环境变量，只需在 Python 脚本中 <code>import inductor_npu_ext</code> 即可使能 AutoFuse。下面主要介绍 GE 路线的配置方式。

### 2.1 最小开启配置

AutoFuse 的基础使能入口是环境变量 <code>AUTOFUSE_FLAGS</code>，最小只需一行命令即可开启：

```bash
export AUTOFUSE_FLAGS="--enable_autofuse=true"
```

<table align="left">
  <thead>
    <tr>
      <th>配置项</th>
      <th>含义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>--enable_autofuse=true</code></td>
      <td>打开 AutoFuse 总开关，允许图编译流程执行自动融合</td>
    </tr>
    <tr>
      <td><code>--enable_autofuse=false</code></td>
      <td>关闭 AutoFuse 总开关，不执行自动融合</td>
    </tr>
  </tbody>
</table>
<div style="clear:left"></div>

设置后需要在 **同一个 shell 环境** 中执行模型编译命令。环境变量不会自动影响已经启动的其他进程。


### 2.2 配置格式说明

如需追加其他配置项，请注意：

- 多个配置项之间用 **英文分号** <code>;</code> 分隔
- 全部写在 **同一个** <code>AUTOFUSE_FLAGS</code> 字符串中

```bash
# 格式示意
export AUTOFUSE_FLAGS="配置项1;配置项2;配置项3"
```

### 2.3 扩展开关配置

AutoFuse 框架目前支持 Elemwise、Broadcast、Reduce、Concat 等 4 类算子的融合。开启 <code>--enable_autofuse=true</code> 后：

- **Elemwise、Broadcast** 等基础融合能力默认可用，无需额外配置
- **Reduce、Concat** 等融合能力默认不开启，需通过 <code>--autofuse_enable_pass</code> 显式声明

例如同时开启 Reduce 和 Concat：

```bash
export AUTOFUSE_FLAGS="--enable_autofuse=true;--autofuse_enable_pass=reduce,concat"
```

<table align="left">
  <thead>
    <tr>
      <th>配置项</th>
      <th>含义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>--autofuse_enable_pass=reduce,concat</code></td>
      <td>额外开启指定融合 pass，多种类型用英文逗号分隔</td>
    </tr>
  </tbody>
</table>
<div style="clear:left"></div>

各融合类型的默认开关状态如下：

<table align="left">
  <thead>
    <tr>
      <th>融合类型</th>
      <th>含义</th>
      <th>默认状态</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Elemwise</td>
      <td>逐元素计算，如 Add、Mul、Abs</td>
      <td>✅ 开启 AutoFuse 后默认支持</td>
    </tr>
    <tr>
      <td>Broadcast</td>
      <td>广播计算，从尾对齐，长度1的维度自动扩展</td>
      <td>✅ 开启 AutoFuse 后默认支持</td>
    </tr>
    <tr>
      <td>Reduce</td>
      <td>规约计算，如 Sum、Max、Mean</td>
      <td>❌ 需显式开启</td>
    </tr>
    <tr>
      <td>Concat</td>
      <td>拼接计算，按指定维度拼接 Tensor</td>
      <td>❌ 需显式开启</td>
    </tr>
  </tbody>
</table>
<div style="clear:left"></div>


### 2.4 配置检查

配置完成后，可通过以下命令确认当前 shell 中是否已经设置 <code>AUTOFUSE_FLAGS</code>：

```bash
echo $AUTOFUSE_FLAGS
```

如果输出为空，说明当前 shell 未设置该变量；如果能看到 <code>--enable_autofuse=true</code> 等配置内容，说明基础开关已设置到当前环境。需要注意，该检查只能确认环境变量是否设置，最终是否产生融合还需结合模型图结构、算子类型、编译日志和性能分析判断。


## 3. 常见误区

<table align="left">
  <thead>
    <tr>
      <th>误区</th>
      <th>正确认知</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>只设置 <code>AUTOFUSE_DFX_FLAGS</code> 就能开启 AutoFuse</td>
      <td><code>AUTOFUSE_DFX_FLAGS</code> 是进阶定位配置，基础开启依赖 <code>AUTOFUSE_FLAGS="--enable_autofuse=true"</code></td>
    </tr>
    <tr>
      <td>设置环境变量后所有模型一定收益</td>
      <td>收益需要结合实际模型或分析执行结果判断</td>
    </tr>
    <tr>
      <td>Reduce、Concat 只要打开总开关就一定融合</td>
      <td>默认不使能的融合能力，需要显式开启</td>
    </tr>
    <tr>
      <td>AutoFuse 是业务代码里直接调用的 API</td>
      <td>它内嵌在 GE 等组件的图编译流程中，用户只需通过环境变量开启即可</td>
    </tr>
  </tbody>
</table>
<div style="clear:left"></div>


## 4. 课后练习

本节介绍了 AutoFuse 的基础使能方式，请根据学习内容完成以下题目进行自测。

1. （判断题）AutoFuse 的基础使能入口是 <code>AUTOFUSE_FLAGS</code>，最小配置可以写为 <code>--enable_autofuse=true</code>。

2. （判断题）<code>AUTOFUSE_DFX_FLAGS</code> 是开启 AutoFuse 的必需配置，不配置该环境变量时 AutoFuse 一定无法开启。

3. （单选题）如果需要额外开启 Reduce 和 Concat 融合能力，以下哪种配置更符合课程中的说明？
    A. <code>export AUTOFUSE_FLAGS="--enable_autofuse=true;--autofuse_enable_pass=reduce,concat"</code>
    B. <code>export AUTOFUSE_FLAGS="--enable_autofuse=false"</code>
    C. <code>export AUTOFUSE_DFX_FLAGS="--debug_dir=/tmp"</code>
    D. <code>export AUTOFUSE_FLAGS="--autofuse_disable_pass=reduce,concat"</code>

4. （单选题）设置 <code>AUTOFUSE_FLAGS</code> 后，为什么仍建议确认当前 shell 中的环境变量？
    A. 环境变量只对当前 shell 或其子进程生效
    B. 环境变量会自动修改所有历史终端
    C. 环境变量设置后会自动持久化到系统中，无需重新配置
    D. 环境变量对所有已打开的终端都自动生效，只需设置一次即可

5. （多选题）从普通使用者视角看，配置 AutoFuse 基础开关时需要注意哪些事项？
    A. 先开启 <code>--enable_autofuse=true</code>
    B. Reduce、Concat 默认不使能，需要时可通过 <code>--autofuse_enable_pass</code> 显式开启
    C. 开启 AutoFuse 后仍要结合模型和 profiling 判断是否收益
    D. 只设置 <code>AUTOFUSE_DFX_FLAGS</code> 就能替代基础开关

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/01.03_answer.txt